In [17]:
import spacy
import numpy as np

nlp = spacy.load("en_core_web_sm")

def avg_embed(sent):
  doc = nlp(sent)
  vect = [token.vector for token in doc if token.has_vector]


  return np.mean(vect, axis=0) if vect else np.zeros((nlp.vocab.vect_length,))

sent = "the dog runs fast"
embedding = avg_embed(sent)
print(embedding[:5])

[ 0.22581619 -0.11688715 -0.403504   -0.08471708 -0.05002382]


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
# Sample corpus

corpus=[
    "The dog runs fast",
    "A cat sleeps on the sofa",
    "Dogs and Cats are great pets"
]

# compute tf-idf scores
vectorizer = TfidfVectorizer()
vectorizer.fit(corpus)
tfidf_scores = vectorizer.transform([sentence]).toarray()[0]
words = vectorizer.get_feature_names_out()

In [20]:
tfidf_scores

array([0.        , 0.        , 0.        , 0.        , 0.52863461,
       0.        , 0.52863461, 0.        , 0.        , 0.        ,
       0.52863461, 0.        , 0.        , 0.40204024])

In [21]:
words

array(['and', 'are', 'cat', 'cats', 'dog', 'dogs', 'fast', 'great', 'on',
       'pets', 'runs', 'sleeps', 'sofa', 'the'], dtype=object)

In [22]:
# Compute weighted embeddings
def tfidf_weighted_embedding(Sentence):
  doc = nlp(sentence)
  word_embeddings = []
  weights = []

  for token in doc:
    word = token.text.lower()
    if word in words and token.has_vector:
      idx= list(words).index(word)
      word_embeddings.append(token.vector * tfidf_scores[idx])
      weights.append(tfidf_scores[idx])
  return np.sum(word_embeddings, axis=0) / np.sum(weights) if weights else np.zeros((nlp.vocab.vect_length,))

embedding = tfidf_weighted_embedding(sentence)
print(embedding[:5])

[ 0.19531656 -0.09783773 -0.43420085 -0.13278283 -0.02624964]


In [23]:
from sklearn.decomposition import PCA

sentence_embedding = np.array([avg_embed(sent) for sent in corpus])
pca = PCA()
pca_result = pca.fit_transform(sentence_embedding)

In [24]:
print(pca_result)

[[ 1.3942733e+00 -1.3193356e+00 -2.1210514e-07]
 [ 8.4134489e-01  1.5564188e+00 -2.1210518e-07]
 [-2.2356181e+00 -2.3708390e-01 -2.1210515e-07]]


In [25]:
# Sentence transformer
from sentence_transformers import SentenceTransformer

In [26]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [28]:
sent= "The dog runs fast"
embedding = model.encode(sent)
print(embedding)

[ 3.23461033e-02  3.10139749e-02  1.02430573e-02  3.83050628e-02
  1.55679183e-03  2.49015242e-02 -9.99559164e-02 -6.27648085e-02
  2.06424724e-02 -5.48851453e-02  1.84693392e-02 -7.99516495e-03
 -2.74904463e-02 -2.85768062e-02  4.75995848e-03 -8.94896314e-02
  1.57573447e-02 -1.99315161e-03  3.89552995e-04 -4.10484597e-02
 -1.85492318e-02 -4.36293669e-02  6.21439852e-02 -1.91383806e-04
 -9.56192166e-02 -2.31770682e-03 -2.86175888e-02 -3.92044634e-02
 -2.42305323e-02 -7.86078423e-02 -3.35992500e-02 -8.01753998e-02
 -1.38001135e-02  1.03922613e-01 -4.96925339e-02 -3.18526290e-02
  3.80634032e-02  1.33414231e-02  1.11100733e-01  1.12295449e-01
  7.18320906e-02 -4.53683287e-02  3.30410637e-02 -3.34733687e-02
 -2.30524391e-02 -3.63449124e-03 -7.72476420e-02 -3.10249608e-02
  2.37952918e-02 -3.66390608e-02 -9.06494781e-02  3.08427750e-03
 -2.18857303e-02 -1.06081320e-02 -2.41962578e-02  3.59360054e-02
 -9.34288092e-03  6.24947362e-02 -1.74000710e-02 -7.09915608e-02
 -1.03087518e-02  5.12834

In [29]:
# NLP Task - Sentence Similarity
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
def compute_similarity(sent1, sent2):
  embedding1 = model.encode(sent1)
  embedding2 = model.encode(sent2)
  similarity_score =  cosine_similarity([embedding1], [embedding2])[0][0]

  return similarity_score

In [31]:
sent1="A dog is running in the park"
sent2="A pup is playing in the park"

In [32]:
similarity = compute_similarity(sent1, sent2)
print(f"Sentence similarity score: {similarity:.4f}")

Sentence similarity score: 0.8162


In [35]:
sent1="Donald Trump is a billionaire"
sent2="Donald Trump is the president of the USA"

In [36]:
similarity = compute_similarity(sent1, sent2)
print(f"Sentence similarity score: {similarity:.4f}")

Sentence similarity score: 0.6624
